In [15]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score


from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix)

In [4]:
TH_500 = pd.read_csv(
    "../classification/TomsHardware/Relative_labeling/sigma=500/TomsHardware-Relative-Sigma-500.data",
    sep=",",
    header=None
)

TH_1000= pd.read_csv(
    "../classification/TomsHardware/Relative_labeling/sigma=1000/TomsHardware-Relative-Sigma-1000.data",
    sep=",",
    header=None
)
TH_1500= pd.read_csv(
    "../classification/TomsHardware/Relative_labeling/sigma=1500/TomsHardware-Relative-Sigma-1500.data",
    sep=",",
    header=None
)
groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = []
for group in groups:
    for t in range(8):
        columns.append(f"{group}_{t}")

columns.append("label")  

TH_500.columns = columns
TH_1000.columns = columns
TH_1500.columns = columns

prefixes = {col.split("_")[0] for col in TH_500.columns if "_" in col}
prefixes = {col.split("_")[0] for col in TH_1000.columns if "_" in col}
prefixes = {col.split("_")[0] for col in TH_1500.columns if "_" in col}

In [7]:
### Dataset sigma=500
top_features = ['ND', 'AS_NA', 'NAC', 'NAD', 'NCD', 'label']

selected_columns = [
    col for col in TH_500.columns 
    if any(col.startswith(f + '_') for f in top_features)
]


Selected_500 = TH_500[selected_columns]
Selected_500['label']=TH_500['label']

print("Number of features:", len(selected_columns))
#Selected_500.head()

### Dataset sigma=1000
top_features = ['ND', 'AS_NA', 'NA', 'NAC', 'AI']

selected_columns = [
    col for col in TH_1000.columns 
    if any(col.startswith(f + '_') for f in top_features)
]

Selected_1000 = TH_1000[selected_columns]
Selected_1000['label']=TH_1000['label']


print("Number of features:", len(selected_columns))
#Selected_1000.head()

### Dataset sigma=1500
top_features = ['ND', 'AS_NA', 'NAC', 'AI', 'AS_NAC']

selected_columns = [
    col for col in TH_1500.columns 
    if any(col.startswith(f + '_') for f in top_features)
]

Selected_1500 = TH_1500[selected_columns]
Selected_1500['label']=TH_1500['label']

print("Number of features:", len(selected_columns))
#Selected_1500.head()

Number of features: 40
Number of features: 40
Number of features: 40


### 500

### priprava

In [8]:
X = Selected_500.drop(columns=['label'])
y = Selected_500['label']

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, stratify=y,random_state=42)

### 1.Random Forest with Grid Search

In [10]:
rf = RandomForestClassifier(random_state=42)

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

print("Random Forest with Grid Search")
print(best_rf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Random Forest with Grid Search
RandomForestClassifier(n_estimators=200, random_state=42)
Accuracy: 0.9259962049335864
Precision: 0.8181818181818182
Recall: 0.853448275862069
F1: 0.8354430379746836
ROC-AUC: 0.9697308685478835


### XGBoost with Grid Search

In [17]:
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

param_grid_xgb = {
    'n_estimators': [100],
    'max_depth': [6],
    'learning_rate': [0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0]
}
grid_xgb = GridSearchCV(xgb,param_grid_xgb, cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_xgb = grid_xgb.best_estimator_
y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:,1]

print("XGBoost with Grid Search")
print(best_xgb)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

XGBoost with Grid Search
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)
Accuracy: 0.9285262492093611
Precision: 0.8291316526610645
Recall: 0.8505747126436781
F1: 0.8397163120567376
ROC-AUC: 0.9722804858722301


In [9]:
### XGBoost randomizer

In [16]:

xgb = XGBClassifier(random_state=42,eval_metric='logloss',use_label_encoder=False)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.3, 0.5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

random_search = RandomizedSearchCV(xgb, param_distributions=param_dist, n_iter=30,  scoring='f1', cv=5,verbose=1,random_state=42,n_jobs=-1).fit(X_train, y_train)

best_xgb_rand = random_search.best_estimator_

print("Best parameters:", random_search.best_params_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters: {'subsample': 0.9, 'reg_lambda': 2, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 1.0}


### Logistic Regression with Scaling and Grid Search

In [17]:
pipe_lr = Pipeline([('scaler', StandardScaler()),('lr', LogisticRegression(solver='liblinear', max_iter=1000))])
param_grid_lr = {
    'lr__C': [0.1, 1, 10],
    'lr__penalty': ['l1', 'l2']
}

grid_lr = GridSearchCV(pipe_lr,param_grid_lr,cv=5,scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_lr = grid_lr.best_estimator_
y_pred = best_lr.predict(X_test)
y_prob = best_lr.predict_proba(X_test)[:,1]

print("Logistic Regression with Scaling and Grid Search")
print(best_lr)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Logistic Regression with Scaling and Grid Search
Pipeline(steps=[('scaler', StandardScaler()),
                ('lr',
                 LogisticRegression(C=10, max_iter=1000, penalty='l1',
                                    solver='liblinear'))])
Accuracy: 0.8969006957621758
Precision: 0.8599221789883269
Recall: 0.6350574712643678
F1: 0.7305785123966942
ROC-AUC: 0.9475603844468682


In [24]:
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True))
])

param_grid_svm = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['rbf', 'linear'],
    'svm__gamma': ['scale', 'auto']
}

grid_svm = GridSearchCV( pipe_svm, param_grid_svm, cv=5, scoring='f1',n_jobs=-1).fit(X_train, y_train)

best_svm = grid_svm.best_estimator_
y_pred = best_svm.predict(X_test)
y_prob = best_svm.predict_proba(X_test)[:,1]

print("SVM")
print("Best SVM:", grid_svm.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

SVM
Best SVM: {'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Accuracy: 0.9209361163820367
Precision: 0.8495297805642633
Recall: 0.7787356321839081
F1: 0.8125937031484258
ROC-AUC: 0.9681262876266652


### STACKING MODEL with logistic regression

In [21]:

estimators = [
    ('rf', best_rf),
    ('xgb', best_xgb),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(estimators=estimators, final_estimator=meta_model,cv=5, n_jobs=-1).fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]


print("STACKING MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING MODEL
Accuracy: 0.92662871600253
Precision: 0.8372093023255814
Recall: 0.8275862068965517
F1: 0.8323699421965318
ROC-AUC: 0.9730658798743369


### STACKING MODEL with logistic regression

In [18]:
estimators = [
    ('rf', best_rf),
    ('xgb', best_xgb_rand ),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(estimators=estimators, final_estimator=meta_model,cv=5, n_jobs=-1).fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]


print("STACKING MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING MODEL
Accuracy: 0.9291587602783049
Precision: 0.8450292397660819
Recall: 0.8304597701149425
F1: 0.8376811594202899
ROC-AUC: 0.9730938464263407


### STACKING MODEL with svm

In [25]:
estimators = [
    ('rf', best_rf),
    ('svm', best_svm),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier( estimators=estimators,final_estimator=meta_model, cv=5,n_jobs=-1, stack_method='predict_proba'  )
stack_model.fit(X_train, y_train)


y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:, 1]

# оцінка
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("STACKING (RF + SVM + LR)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING (RF + SVM + LR)
Accuracy: 0.9234661606578115
Precision: 0.8270893371757925
Recall: 0.8247126436781609
F1: 0.8258992805755395
ROC-AUC: 0.9720590840022


### 1000

### priprava

In [19]:
X = Selected_1000.drop(columns=['label'])
y = Selected_1000['label']

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, stratify=y,random_state=42)

### 1.Random Forest with Grid Search

In [20]:
rf = RandomForestClassifier(random_state=42)

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

print("Random Forest with Grid Search")
print(best_rf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Random Forest with Grid Search
RandomForestClassifier(n_estimators=200, random_state=42)
Accuracy: 0.9259962049335864
Precision: 0.8181818181818182
Recall: 0.853448275862069
F1: 0.8354430379746836
ROC-AUC: 0.9697308685478835


### XGBoost with Grid Search

In [21]:
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

param_grid_xgb = {
    'n_estimators': [100],
    'max_depth': [6],
    'learning_rate': [0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0]
}
grid_xgb = GridSearchCV(xgb,param_grid_xgb, cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_xgb = grid_xgb.best_estimator_
y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:,1]

print("XGBoost with Grid Search")
print(best_xgb)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

XGBoost with Grid Search
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)
Accuracy: 0.9285262492093611
Precision: 0.8291316526610645
Recall: 0.8505747126436781
F1: 0.8397163120567376
ROC-AUC: 0.9722804858722301


### XGBoost randomizer

In [22]:
xgb = XGBClassifier(random_state=42,eval_metric='logloss',use_label_encoder=False)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.3, 0.5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

random_search = RandomizedSearchCV(xgb, param_distributions=param_dist, n_iter=30,  scoring='f1', cv=5,verbose=1,random_state=42,n_jobs=-1).fit(X_train, y_train)

best_xgb_rand = random_search.best_estimator_

print("Best parameters:", random_search.best_params_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters: {'subsample': 0.9, 'reg_lambda': 2, 'reg_alpha': 0.1, 'n_estimators': 200, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0.1, 'colsample_bytree': 1.0}


### Logistic Regression with Scaling and Grid Search

In [23]:
pipe_lr = Pipeline([('scaler', StandardScaler()),('lr', LogisticRegression(solver='liblinear', max_iter=1000))])
param_grid_lr = {
    'lr__C': [0.1, 1, 10],
    'lr__penalty': ['l1', 'l2']
}

grid_lr = GridSearchCV(pipe_lr,param_grid_lr,cv=5,scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_lr = grid_lr.best_estimator_
y_pred = best_lr.predict(X_test)
y_prob = best_lr.predict_proba(X_test)[:,1]

print("Logistic Regression with Scaling and Grid Search")
print(best_lr)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Logistic Regression with Scaling and Grid Search
Pipeline(steps=[('scaler', StandardScaler()),
                ('lr',
                 LogisticRegression(C=10, max_iter=1000, penalty='l1',
                                    solver='liblinear'))])
Accuracy: 0.8975332068311196
Precision: 0.8604651162790697
Recall: 0.6379310344827587
F1: 0.7326732673267327
ROC-AUC: 0.9475580539008679


### svm

In [26]:
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True))
])

param_grid_svm = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['rbf', 'linear'],
    'svm__gamma': ['scale', 'auto']
}

grid_svm = GridSearchCV( pipe_svm, param_grid_svm, cv=5, scoring='f1',n_jobs=-1).fit(X_train, y_train)

best_svm = grid_svm.best_estimator_
y_pred = best_svm.predict(X_test)
y_prob = best_svm.predict_proba(X_test)[:,1]

print("SVM")
print("Best SVM:", grid_svm.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

SVM
Best SVM: {'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Accuracy: 0.9209361163820367
Precision: 0.8495297805642633
Recall: 0.7787356321839081
F1: 0.8125937031484258
ROC-AUC: 0.9681297834456657


### STACKING MODEL with logistic regression

In [27]:

estimators = [
    ('rf', best_rf),
    ('xgb', best_xgb),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(estimators=estimators, final_estimator=meta_model,cv=5, n_jobs=-1).fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]


print("STACKING MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING MODEL
Accuracy: 0.92662871600253
Precision: 0.8372093023255814
Recall: 0.8275862068965517
F1: 0.8323699421965318
ROC-AUC: 0.9730635493283366


### STACKING MODEL with logistic regression

In [29]:
estimators = [
    ('rf', best_rf),
    ('xgb', best_xgb_rand ),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(estimators=estimators, final_estimator=meta_model,cv=5, n_jobs=-1).fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]


print("STACKING MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING MODEL
Accuracy: 0.9291587602783049
Precision: 0.8450292397660819
Recall: 0.8304597701149425
F1: 0.8376811594202899
ROC-AUC: 0.973096176972341


### STACKING MODEL with svm

In [30]:
estimators = [
    ('rf', best_rf),
    ('svm', best_svm),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier( estimators=estimators,final_estimator=meta_model, cv=5,n_jobs=-1, stack_method='predict_proba'  )
stack_model.fit(X_train, y_train)


y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:, 1]

# оцінка
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("STACKING (RF + SVM + LR)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING (RF + SVM + LR)
Accuracy: 0.9234661606578115
Precision: 0.8270893371757925
Recall: 0.8247126436781609
F1: 0.8258992805755395
ROC-AUC: 0.9720544229101994


рамках дослідження було побудовано кілька ансамблевих моделей stacking із використанням різних базових алгоритмів, зокрема Random Forest, XGBoost, Logistic Regression та SVM.

Найкращі результати серед окремих моделей продемонстрував XGBoost після оптимізації гіперпараметрів (F1-score = 0.840, ROC-AUC = 0.972), що підтверджує його високу ефективність для задачі класифікації з табличними даними. Random Forest показав подібні результати (F1 = 0.835), тоді як Logistic Regression та SVM продемонстрували нижчу якість, особливо за показником recall.

Побудова stacking-моделі з використанням XGBoost (після Grid Search) дозволила досягти F1-score = 0.832, що є дещо нижчим за найкращу базову модель. Однак використання більш оптимізованої версії XGBoost (Randomized Search) у складі ансамблю дало покращення до F1-score = 0.838 та ROC-AUC = 0.973, що практично відповідає найкращим результатам серед усіх моделей.

Водночас альтернативний stacking без XGBoost (RF + SVM + LR) показав нижчу ефективність (F1 = 0.826), що свідчить про важливу роль XGBoost як ключового компонента ансамблю.

Отримані результати демонструють, що stacking не завжди гарантує суттєве покращення порівняно з найкращою базовою моделлю. У даному випадку ансамбль лише незначно покращує баланс між precision і recall, але не перевершує XGBoost як окрему модель.

### 1500


### priprava

In [31]:
X = Selected_1500.drop(columns=['label'])
y = Selected_1500['label']

X_train, X_test, y_train, y_test = train_test_split( X, y, test_size=0.2, stratify=y,random_state=42)

### 1.Random Forest with Grid Search

In [32]:
rf = RandomForestClassifier(random_state=42)

param_grid_rf = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt']
}

grid_rf = GridSearchCV(rf, param_grid_rf, cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_rf = grid_rf.best_estimator_

y_pred = best_rf.predict(X_test)
y_prob = best_rf.predict_proba(X_test)[:,1]

print("Random Forest with Grid Search")
print(best_rf)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Random Forest with Grid Search
RandomForestClassifier(random_state=42)
Accuracy: 0.9639468690702088
Precision: 0.8627450980392157
Recall: 0.7857142857142857
F1: 0.822429906542056
ROC-AUC: 0.9871516193172245


### XGBoost with Grid Search

In [34]:
xgb = XGBClassifier(random_state=42, use_label_encoder=False, eval_metric='logloss')

param_grid_xgb = {
    'n_estimators': [100],
    'max_depth': [6],
    'learning_rate': [0.1],
    'subsample': [0.8],
    'colsample_bytree': [0.8, 1.0]
}
grid_xgb = GridSearchCV(xgb,param_grid_xgb, cv=5, scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_xgb = grid_xgb.best_estimator_
y_pred = best_xgb.predict(X_test)
y_prob = best_xgb.predict_proba(X_test)[:,1]

print("XGBoost with Grid Search")
print(best_xgb)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

XGBoost with Grid Search
XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=1.0, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=6, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)
Accuracy: 0.9626818469323213
Precision: 0.8657718120805369
Recall: 0.7678571428571429
F1: 0.8138801261829653
ROC-AUC: 0.9889124793583393


### XGBoost randomizer

In [36]:

xgb = XGBClassifier(random_state=42,eval_metric='logloss',use_label_encoder=False)

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [3, 5, 6, 8, 10],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5, 7],
    'gamma': [0, 0.1, 0.3, 0.5],
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [1, 1.5, 2]
}

random_search = RandomizedSearchCV(xgb, param_distributions=param_dist, n_iter=30,  scoring='f1', cv=5,verbose=1,random_state=42,n_jobs=-1).fit(X_train, y_train)

best_xgb_rand = random_search.best_estimator_

print("Best parameters:", random_search.best_params_)

Fitting 5 folds for each of 30 candidates, totalling 150 fits
Best parameters: {'subsample': 0.9, 'reg_lambda': 1, 'reg_alpha': 0, 'n_estimators': 300, 'min_child_weight': 1, 'max_depth': 6, 'learning_rate': 0.1, 'gamma': 0.3, 'colsample_bytree': 1.0}


### Logistic Regression with Scaling and Grid Search

In [37]:
pipe_lr = Pipeline([('scaler', StandardScaler()),('lr', LogisticRegression(solver='liblinear', max_iter=1000))])
param_grid_lr = {
    'lr__C': [0.1, 1, 10],
    'lr__penalty': ['l1', 'l2']
}

grid_lr = GridSearchCV(pipe_lr,param_grid_lr,cv=5,scoring='f1', n_jobs=-1).fit(X_train, y_train)

best_lr = grid_lr.best_estimator_
y_pred = best_lr.predict(X_test)
y_prob = best_lr.predict_proba(X_test)[:,1]

print("Logistic Regression with Scaling and Grid Search")
print(best_lr)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Logistic Regression with Scaling and Grid Search
Pipeline(steps=[('scaler', StandardScaler()),
                ('lr',
                 LogisticRegression(C=10, max_iter=1000, penalty='l1',
                                    solver='liblinear'))])
Accuracy: 0.9538266919671095
Precision: 0.8925619834710744
Recall: 0.6428571428571429
F1: 0.7474048442906575
ROC-AUC: 0.970073804468709


### svm

In [39]:
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(probability=True))
])

param_grid_svm = {
    'svm__C': [0.1, 1, 10],
    'svm__kernel': ['rbf', 'linear'],
    'svm__gamma': ['scale', 'auto']
}

grid_svm = GridSearchCV( pipe_svm, param_grid_svm, cv=5, scoring='f1',n_jobs=-1).fit(X_train, y_train)

best_svm = grid_svm.best_estimator_
y_pred = best_svm.predict(X_test)
y_prob = best_svm.predict_proba(X_test)[:,1]

print("SVM")
print("Best SVM:", grid_svm.best_params_)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

SVM
Best SVM: {'svm__C': 10, 'svm__gamma': 'scale', 'svm__kernel': 'rbf'}
Accuracy: 0.9645793801391525
Precision: 0.9057971014492754
Recall: 0.7440476190476191
F1: 0.8169934640522876
ROC-AUC: 0.9865492198294746


### STACKING MODEL with logistic regression

In [40]:

estimators = [
    ('rf', best_rf),
    ('xgb', best_xgb),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(estimators=estimators, final_estimator=meta_model,cv=5, n_jobs=-1).fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]


print("STACKING MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING MODEL
Accuracy: 0.9607843137254902
Precision: 0.8785714285714286
Recall: 0.7321428571428571
F1: 0.7987012987012987
ROC-AUC: 0.9884912209752974


### STACKING MODEL with logistic regression

In [42]:
estimators = [
    ('rf', best_rf),
    ('xgb', best_xgb_rand ),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier(estimators=estimators, final_estimator=meta_model,cv=5, n_jobs=-1).fit(X_train, y_train)

y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:,1]


print("STACKING MODEL")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING MODEL
Accuracy: 0.9607843137254902
Precision: 0.8785714285714286
Recall: 0.7321428571428571
F1: 0.7987012987012987
ROC-AUC: 0.9885207090621103


### STACKING MODEL with svm

In [43]:
estimators = [
    ('rf', best_rf),
    ('svm', best_svm),
    ('lr', best_lr)
]

meta_model = LogisticRegression()

stack_model = StackingClassifier( estimators=estimators,final_estimator=meta_model, cv=5,n_jobs=-1, stack_method='predict_proba'  )
stack_model.fit(X_train, y_train)


y_pred = stack_model.predict(X_test)
y_prob = stack_model.predict_proba(X_test)[:, 1]

# оцінка
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

print("STACKING (RF + SVM + LR)")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

STACKING (RF + SVM + LR)
Accuracy: 0.9620493358633776
Precision: 0.8913043478260869
Recall: 0.7321428571428571
F1: 0.803921568627451
ROC-AUC: 0.9884617328884845


У ході дослідження було побудовано ансамблеві моделі stacking для датасету TH_1500, який характеризується найвищим рівнем незбалансованості серед усіх розглянутих наборів даних.

Серед окремих моделей найкращі результати продемонстрував Random Forest (F1-score = 0.822) та SVM (F1 = 0.817), які забезпечили оптимальний баланс між precision та recall. XGBoost показав дещо нижчий результат (F1 = 0.814), хоча досяг найвищого значення ROC-AUC (0.989), що свідчить про хорошу здатність моделі до ранжування об’єктів.

Logistic Regression продемонструвала найнижчі результати серед усіх моделей (F1 = 0.747), що пов’язано з низьким значенням recall, незважаючи на високу precision.

Побудовані stacking-моделі не продемонстрували покращення якості класифікації. Основна stacking-модель (з XGBoost) досягла F1-score = 0.799, що є нижчим за результати найкращих базових моделей. Альтернативний ансамбль без XGBoost (RF + SVM + LR) показав незначно кращий результат (F1 = 0.804), однак також поступився Random Forest як окремій моделі.

Отримані результати свідчать про те, що у випадку більш незбалансованих даних stacking не забезпечує покращення продуктивності. Це може бути пов’язано з тим, що базові моделі вже добре узагальнюють дані, а їх поєднання не додає нової інформації, а лише ускладнює модель.

In [1]:
import pandas as pd

data = [
    ["Stacking (500)", 0.9266, 0.8372, 0.8276, 0.8324, 0.9731],
    ["Stacking (1000)", 0.9266, 0.8372, 0.8276, 0.8324, 0.9731],
    ["Stacking (1500)", 0.9608, 0.8786, 0.7321, 0.7987, 0.9885],
]

columns = ["Model", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]

df = pd.DataFrame(data, columns=columns)

df

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Stacking (500),0.9266,0.8372,0.8276,0.8324,0.9731
1,Stacking (1000),0.9266,0.8372,0.8276,0.8324,0.9731
2,Stacking (1500),0.9608,0.8786,0.7321,0.7987,0.9885


In [2]:
import pandas as pd

data = [
    ["Stacking (500)", 0.9292, 0.8450, 0.8305, 0.8377, 0.9731],
    ["Stacking (1000)", 0.9292, 0.8450, 0.8305, 0.8377, 0.9731],
    ["Stacking (1500)", 0.9608, 0.8786, 0.7321, 0.7987, 0.9885],
]

columns = ["Model", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]

df = pd.DataFrame(data, columns=columns)

df

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Stacking (500),0.9292,0.8450,0.8305,0.8377,0.9731
1,Stacking (1000),0.9292,0.8450,0.8305,0.8377,0.9731
2,Stacking (1500),0.9608,0.8786,0.7321,0.7987,0.9885


In [4]:
import pandas as pd

data = [
    ["Stacking (500)", 0.9235, 0.8271, 0.8247, 0.8259, 0.9721],
    ["Stacking (1000)", 0.9235, 0.8271, 0.8247, 0.8259, 0.9721],
    ["Stacking (1500)", 0.9620, 0.8913, 0.7321, 0.8039, 0.9885],
]

columns = ["Model", "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC"]

df = pd.DataFrame(data, columns=columns)

df

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Stacking (500),0.9235,0.8271,0.8247,0.8259,0.9721
1,Stacking (1000),0.9235,0.8271,0.8247,0.8259,0.9721
2,Stacking (1500),0.9620,0.8913,0.7321,0.8039,0.9885
